# A2 Mahim: WebQA-Adv Retrieval + Verification

Objective:
- Implement BM25 retrieval over full WebQA text pool.
- Fine-tune RoBERTa NLI for 3-way verification (TRUE/FALSE/UNVERIFIABLE).
- Compare oracle evidence vs BM25-retrieved evidence.
- Produce required analyses and figures.

Done criteria:
- Recall@5/10/20 reported (+ curve with K=1/5/10/20).
- RoBERTa trained with AdamW, LR=2e-5, warmup, max_len=512, class-weighted CE.
- Confusion matrices (oracle + end-to-end), per-class performance, oracle-vs-end-to-end bars.
- Retrieval and verification breakdown by misinformation type.
- 2-3 qualitative successes + failures and attention heatmaps.


## 0) Environment Setup

Install once if needed (HPC login node may require `--user`):
```bash
pip install -U pandas numpy matplotlib seaborn scikit-learn datasets transformers rank_bm25 accelerate torch
```


In [ ]:
from __future__ import annotations

from pathlib import Path
import json
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

import sys
sys.path.append('/share/csc791003s26/cmvn/dataset-folder/scripts')

import a2_pipeline_utils as U

U.set_seed(123)

print('Imports loaded.')


In [ ]:
# Paths and experiment config
ROOT = Path('/share/csc791003s26/cmvn/dataset-folder')
WEBQA_ADV_PATH = ROOT / 'step0_out' / 'webqa_adv.jsonl'
STEP0_PARSED_PATH = ROOT / 'step0_out' / 'webqa_parsed.json'
TEXT_POOL_PATH = ROOT / 'step0_out' / 'text_snippets.jsonl'

OUT_DIR = U.ensure_dir(ROOT / 'a2_outputs')
FIG_DIR = U.ensure_dir(OUT_DIR / 'figures')
MODEL_DIR = U.ensure_dir(OUT_DIR / 'models' / 'roberta_nli')
TABLE_DIR = U.ensure_dir(OUT_DIR / 'tables')

CFG = {
    'seed': 123,
    'bm25_k_values': [1, 5, 10, 20],
    'retrieved_topk_for_verifier': 1,
    'model_name': 'roberta-large-mnli',
    'learning_rate': 2e-5,
    'warmup_ratio': 0.1,
    'batch_size': 16,
    'epochs': 3,
    'max_length': 512,
    'max_train_samples': 18000,   # keep training lighter
    'max_dev_samples': 4000,
    'max_test_samples': 6000,
}

print('ROOT:', ROOT)
print('Output dir:', OUT_DIR)
print(json.dumps(CFG, indent=2))


## 1) Load WebQA-Adv + Step-0 Artifacts

In [ ]:
assert WEBQA_ADV_PATH.exists(), f'Missing: {WEBQA_ADV_PATH}'
assert STEP0_PARSED_PATH.exists(), f'Missing: {STEP0_PARSED_PATH}'
assert TEXT_POOL_PATH.exists(), f'Missing: {TEXT_POOL_PATH}'

adv_df = U.load_webqa_adv_df(WEBQA_ADV_PATH, STEP0_PARSED_PATH)
text_pool_df = U.load_text_pool(TEXT_POOL_PATH)

print('WebQA-Adv rows:', len(adv_df))
print('Text pool rows:', len(text_pool_df))
print('')
print('Split counts:')
print(adv_df['split'].value_counts(dropna=False))
print('')
print('Label counts:')
print(adv_df['label'].value_counts(dropna=False))
print('')
print('Manipulation counts:')
print(adv_df['manipulation_type'].value_counts(dropna=False))

adv_df.head(2)


## 2) Phase 1: BM25 Text Retrieval

In [ ]:
# Build BM25 over full text snippet pool (~540K in full release)
bm25, corpus_ids, corpus_texts, _ = U.build_bm25_index(text_pool_df)
print('BM25 index built. Corpus size:', len(corpus_ids))


In [ ]:
# Retrieve top-K for every claim and compute Recall@K
retrieval_df, recalls = U.retrieve_topk_for_claims(
    adv_df,
    bm25,
    corpus_ids,
    corpus_texts,
    k_values=CFG['bm25_k_values'],
)

merged_retrieval = adv_df.merge(retrieval_df, on='id', how='left')

print('Recall summary:')
for k in CFG['bm25_k_values']:
    print(f'Recall@{k}: {recalls[k]:.4f}')

merged_retrieval[[
    'id', 'claim', 'label', 'manipulation_type',
    'hit@1', 'hit@5', 'hit@10', 'hit@20'
]].head(5)


In [ ]:
# Retrieval analysis by misinformation type and by label
retrieval_by_type = U.recall_by_group(merged_retrieval, 'manipulation_type', CFG['bm25_k_values'])
retrieval_by_label = U.recall_by_group(merged_retrieval, 'label', CFG['bm25_k_values'])

print('Retrieval by manipulation type:')
display(retrieval_by_type)
print('Retrieval by label:')
display(retrieval_by_label)

U.save_table_image(retrieval_by_type, TABLE_DIR / 'retrieval_by_type.png', title='Retrieval Recall by Manipulation Type')
U.save_table_image(retrieval_by_label, TABLE_DIR / 'retrieval_by_label.png', title='Retrieval Recall by Label')


In [ ]:
# Retrieval failure analysis (claims where gold text not found in top-10)
retrieval_failures = U.retrieval_failure_examples(merged_retrieval, k=10, n=15)
retrieval_failures.to_csv(TABLE_DIR / 'retrieval_failures_top10.csv', index=False)
retrieval_failures


In [ ]:
# Figure: Recall@K curve
U.plot_recall_curve(recalls, FIG_DIR / 'recall_at_k_curve.png', title='BM25 Recall@K (WebQA-Adv Claims)')
print('Saved:', FIG_DIR / 'recall_at_k_curve.png')


## 3) Phase 2: RoBERTa NLI Verification (Oracle vs Retrieved)

Training setup:
- Optimizer: AdamW (Transformers Trainer default)
- LR: 2e-5
- Warmup: linear warmup via `warmup_ratio`
- Batch size: 16
- Epochs: 3
- Max length: 512
- Loss: class-weighted cross-entropy


In [ ]:
# Build NLI pairs for two settings:
# 1) Oracle evidence (gold snippet text)
# 2) End-to-end retrieved evidence (BM25 top-1 or top-k concatenated)
oracle_nli_df = U.prepare_nli_dataframe(
    adv_df,
    retrieval_df=retrieval_df,
    mode='oracle',
    top_k=1,
)
retrieved_nli_df = U.prepare_nli_dataframe(
    adv_df,
    retrieval_df=retrieval_df,
    mode='retrieved',
    top_k=CFG['retrieved_topk_for_verifier'],
)

print('Oracle NLI pairs:', len(oracle_nli_df))
print('Retrieved NLI pairs:', len(retrieved_nli_df))

oracle_nli_df.head(2)


In [ ]:
# Split and cap sample sizes for lighter fine-tuning
train_oracle_df, dev_oracle_df, test_oracle_df = U.split_dataframe(oracle_nli_df)
_, _, test_retrieved_df = U.split_dataframe(retrieved_nli_df)

train_oracle_df = U.sample_cap(train_oracle_df, CFG['max_train_samples'], CFG['seed'])
dev_oracle_df = U.sample_cap(dev_oracle_df, CFG['max_dev_samples'], CFG['seed'])
test_oracle_df = U.sample_cap(test_oracle_df, CFG['max_test_samples'], CFG['seed'])
test_retrieved_df = U.sample_cap(test_retrieved_df, CFG['max_test_samples'], CFG['seed'])

print('Train/Dev/Test (oracle):', len(train_oracle_df), len(dev_oracle_df), len(test_oracle_df))
print('Test (retrieved):', len(test_retrieved_df))


In [ ]:
# Fine-tune RoBERTa NLI on oracle evidence (train/dev)
trainer, tokenizer = U.train_roberta_nli(
    train_df=train_oracle_df,
    dev_df=dev_oracle_df,
    model_name=CFG['model_name'],
    out_dir=MODEL_DIR,
    learning_rate=CFG['learning_rate'],
    batch_size=CFG['batch_size'],
    num_epochs=CFG['epochs'],
    warmup_ratio=CFG['warmup_ratio'],
    max_length=CFG['max_length'],
    seed=CFG['seed'],
)
print('Training complete. Best checkpoint:', trainer.state.best_model_checkpoint)


In [ ]:
# Evaluate on oracle test and end-to-end retrieved test
oracle_pred_df = U.predict_with_model(trainer, tokenizer, test_oracle_df, max_length=CFG['max_length'])
e2e_pred_df = U.predict_with_model(trainer, tokenizer, test_retrieved_df, max_length=CFG['max_length'])

oracle_metrics = U.metrics_from_predictions(oracle_pred_df)
e2e_metrics = U.metrics_from_predictions(e2e_pred_df)

print('Oracle -> Accuracy:', round(oracle_metrics['accuracy'], 4), 'Macro-F1:', round(oracle_metrics['macro_f1'], 4))
print('E2E    -> Accuracy:', round(e2e_metrics['accuracy'], 4), 'Macro-F1:', round(e2e_metrics['macro_f1'], 4))

metrics_table = pd.concat([
    U.report_metrics_table(oracle_metrics, 'oracle'),
    U.report_metrics_table(e2e_metrics, 'end_to_end'),
], ignore_index=True)

metrics_table.to_csv(TABLE_DIR / 'oracle_vs_e2e_metrics.csv', index=False)
metrics_table


In [ ]:
# Verification breakdown by misinformation type
oracle_by_type = U.metrics_by_group(oracle_pred_df, 'manipulation_type')
e2e_by_type = U.metrics_by_group(e2e_pred_df, 'manipulation_type')

print('Oracle metrics by type:')
display(oracle_by_type)
print('End-to-end metrics by type:')
display(e2e_by_type)

oracle_by_type.to_csv(TABLE_DIR / 'oracle_by_type.csv', index=False)
e2e_by_type.to_csv(TABLE_DIR / 'e2e_by_type.csv', index=False)


## 4) Required Figures

In [ ]:
# Confusion matrices
U.plot_confusion(oracle_metrics['confusion'], FIG_DIR / 'confusion_oracle.png', 'Confusion Matrix (Oracle Evidence)')
U.plot_confusion(e2e_metrics['confusion'], FIG_DIR / 'confusion_end_to_end.png', 'Confusion Matrix (End-to-End BM25)')

# Per-class performance bar chart
U.plot_per_class_f1(oracle_metrics, e2e_metrics, FIG_DIR / 'per_class_f1_oracle_vs_e2e.png')

# Oracle vs end-to-end comparison bar chart
U.plot_oracle_vs_e2e(oracle_metrics, e2e_metrics, FIG_DIR / 'oracle_vs_e2e_bar.png')

# Training/validation loss curves
train_logs = U.plot_training_curves(trainer, FIG_DIR / 'train_val_loss_curves.png')
train_logs.to_csv(TABLE_DIR / 'training_log_history.csv', index=False)

# Optional confidence distribution (requested as optional in prompt)
U.plot_confidence_distributions(e2e_pred_df, FIG_DIR / 'confidence_distribution_e2e.png')

print('Saved figures to:', FIG_DIR)
sorted([p.name for p in Path(FIG_DIR).glob('*.png')])


In [ ]:
# Additional per-class accuracy (recall) bar chart + macro-F1 marker
classes = ['TRUE', 'FALSE', 'UNVERIFIABLE']
oracle_cls_acc = [oracle_metrics['report'][c]['recall'] for c in classes]
e2e_cls_acc = [e2e_metrics['report'][c]['recall'] for c in classes]

x = np.arange(len(classes))
w = 0.35
plt.figure(figsize=(8, 4.8))
plt.bar(x - w/2, oracle_cls_acc, w, label='Oracle class accuracy (recall)')
plt.bar(x + w/2, e2e_cls_acc, w, label='E2E class accuracy (recall)')
plt.axhline(oracle_metrics['macro_f1'], linestyle='--', color='tab:blue', alpha=0.7, label='Oracle Macro-F1')
plt.axhline(e2e_metrics['macro_f1'], linestyle='--', color='tab:orange', alpha=0.7, label='E2E Macro-F1')
plt.xticks(x, classes)
plt.ylim(0, 1.0)
plt.title('Per-Class Accuracy + Macro-F1')
plt.tight_layout()
plt.legend()
out_path = FIG_DIR / 'per_class_accuracy_macrof1.png'
plt.savefig(out_path, dpi=160)
plt.show()
print('Saved:', out_path)


## 5) Qualitative Analysis

In [ ]:
# 2-3 success and 2-3 failure cases (end-to-end)
succ_df, fail_df = U.qualitative_examples(e2e_pred_df, n_success=3, n_fail=3, seed=CFG['seed'])

succ_df.to_csv(TABLE_DIR / 'qualitative_success_cases.csv', index=False)
fail_df.to_csv(TABLE_DIR / 'qualitative_failure_cases.csv', index=False)

print('Success cases:')
display(succ_df)
print('Failure cases:')
display(fail_df)


In [ ]:
# Attention heatmaps for one success + one failure example
# Note: attention visualization can be noisy; treat as qualitative.

if len(succ_df) > 0:
    s = succ_df.iloc[0]
    U.attention_heatmap(
        model=trainer.model,
        tokenizer=tokenizer,
        evidence_text=str(s['evidence_text']),
        claim=str(s['claim']),
        out_path=FIG_DIR / 'attention_success.png',
        layer=-1,
        head=0,
        max_tokens=60,
    )

if len(fail_df) > 0:
    f = fail_df.iloc[0]
    U.attention_heatmap(
        model=trainer.model,
        tokenizer=tokenizer,
        evidence_text=str(f['evidence_text']),
        claim=str(f['claim']),
        out_path=FIG_DIR / 'attention_failure.png',
        layer=-1,
        head=0,
        max_tokens=60,
    )

print('Saved attention heatmaps:')
for name in ['attention_success.png', 'attention_failure.png']:
    p = FIG_DIR / name
    print(name, '->', p.exists())


## 6) End-to-End vs Oracle Gap (Error Isolation)

Interpretation guideline:
- Oracle performance approximates verifier capacity when evidence is correct.
- End-to-end performance includes retrieval + verification errors.
- The gap quantifies retrieval-induced loss.


In [ ]:
gap = {
    'accuracy_gap': oracle_metrics['accuracy'] - e2e_metrics['accuracy'],
    'macro_f1_gap': oracle_metrics['macro_f1'] - e2e_metrics['macro_f1'],
}

print('Performance gap (oracle - end_to_end):')
print(json.dumps(gap, indent=2))

summary = pd.DataFrame([
    {'setting': 'oracle', 'accuracy': oracle_metrics['accuracy'], 'macro_f1': oracle_metrics['macro_f1']},
    {'setting': 'end_to_end', 'accuracy': e2e_metrics['accuracy'], 'macro_f1': e2e_metrics['macro_f1']},
    {'setting': 'gap', 'accuracy': gap['accuracy_gap'], 'macro_f1': gap['macro_f1_gap']},
])
summary.to_csv(TABLE_DIR / 'summary_oracle_vs_e2e.csv', index=False)
summary


## 7) Deliverables Saved

Files generated under:
- `a2_outputs/figures/*.png`
- `a2_outputs/tables/*.csv`
- `a2_outputs/models/roberta_nli/`

Recommended for report:
1. `recall_at_k_curve.png`
2. `confusion_oracle.png`
3. `confusion_end_to_end.png`
4. `per_class_f1_oracle_vs_e2e.png`
5. `oracle_vs_e2e_bar.png`
6. `train_val_loss_curves.png`
7. `attention_success.png`, `attention_failure.png`
